In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ===============================================================
#
#   九合一新聞爬蟲 (CNA, UDN, SETN, TVBS, EBC, NOWnews, ETtoday, LTN, ChinaTimes)
#   版本: v14 (2025-11-10 - 全新資料集版)
#
#   - 依使用者要求，將輸出檔案改為 'news_dataset.csv'。
#   - 建立新的 'news_dataset_cache.json'，以確保抓取全新資料。
#   - 整合 v13 所有媒體的爬蟲修復。
#
# ===============================================================

# ===============================================================
# Cell 1: 環境設定、全域變數與路徑
# ===============================================================
import sys
import subprocess
import importlib
import logging
import os
from datetime import datetime, timezone, timedelta
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import re
import time
import lxml # 雖然程式碼中可能沒有顯式呼叫 lxml，但 bs4 會使用它來加速解析

# --- 設定路徑 ---
# 讓腳本在 crontab 中也能正確找到檔案
try:
    # __file__ 變數只在 .py 檔案被執行時存在
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # 當在 Jupyter Notebook 或類似環境中執行時，使用 getcwd()
    BASE_DIR = os.getcwd() 

# *** v14 變更：使用新的檔案名稱 ***
DB_FILENAME = os.path.join(BASE_DIR, 'news_dataset.csv')
TEMP_JSON_FILENAME = os.path.join(BASE_DIR, 'news_dataset_cache.json')
LOG_FILENAME = os.path.join(BASE_DIR, 'news_scraper.log') # 日誌檔保持不變

# --- 設定日誌 (Logging) ---
# 移除所有既有的日誌處理器，避免重複輸出
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
    
# 設定日誌記錄：同時輸出到檔案 (LOG_FILENAME) 與終端機
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILENAME, 'a', 'utf-8'), # 'a' = append (附加模式)
        logging.StreamHandler(sys.stdout) # 輸出到終端機
    ]
)

# --- 全域變數 ---
# 統一的 HTTP Headers
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}
# 欄位名稱 (必須完全符合您的 news_dataset.csv 檔案)
SNAPSHOT_COLUMNS = [
    'url', 'title', 'headline_level', 'snapshot_at', 'full_content', 'source', 'url_hash'
]

# ===============================================================
# Cell 2: 通用工具函式 (Helper Functions)
# ===============================================================

def get_current_utc_time():
    """回傳當前的 ISO 8601 格式 UTC 時間字串"""
    return datetime.now(timezone.utc).isoformat()

def normalize_url(base_url, relative_url):
    """將相對 URL 轉換為絕對 URL"""
    if not relative_url:
        return None
    if relative_url.startswith(('http://', 'https://')):
        return relative_url
    if relative_url.startswith('//'):
        return 'https:' + relative_url
    # 處理 base_url 結尾沒有 / 的情況
    if not base_url.endswith('/'):
        base_url += '/'
    # 處理 relative_url 開頭有 / 的情況
    if relative_url.startswith('/'):
        relative_url = relative_url[1:]
    
    # 使用 requests.compat.urljoin 進行更穩健的 URL 組合
    return requests.compat.urljoin(base_url, relative_url)

def get_soup(url, encoding=None):
    """
    發送 HTTP GET 請求並回傳 BeautifulSoup 解析後的物件。
    """
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status() # 如果 HTTP 狀態碼不是 200，拋出例外
        
        if encoding:
            response.encoding = encoding # 強制指定編碼
        else:
            response.encoding = response.apparent_encoding # 自動偵測編碼
            
        soup = BeautifulSoup(response.text, 'lxml')
        return soup
    except requests.exceptions.Timeout:
        logging.warning(f"🟡 請求超時: {url}")
    except requests.exceptions.HTTPError as e:
        logging.warning(f"🟡 HTTP 錯誤 {e.response.status_code}: {url}")
    except requests.exceptions.RequestException as e:
        logging.warning(f"🟡 請求失敗: {url} ({e})")
    except Exception as e:
        logging.error(f"🔥 解析 BeautifulSoup 失敗: {url} ({e})")
        
    return None # 發生任何錯誤都回傳 None

# ===============================================================
# Cell 3: Part 1 核心功能 (抓取頭條 URL)
# ===============================================================

def load_headlines_from_json(filepath=TEMP_JSON_FILENAME):
    """
    從 JSON 檔案載入先前抓取的頭條 URL。
    回傳一個 Set，包含所有已知的 URL。
    """
    try:
        if os.path.exists(filepath):
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                # 確保回傳的是一個 Set，以便快速查找
                return set(data.get('urls', []))
        else:
            logging.info(f"JSON 暫存檔 {filepath} 不存在，將建立新檔案。")
    except json.JSONDecodeError:
        logging.warning(f"🟡 JSON 檔案 {filepath} 格式錯誤，將建立新檔案。")
    except Exception as e:
        logging.error(f"🔥 載入 JSON 檔案時發生錯誤: {e}")
    
    return set() # 發生錯誤或檔案不存在時，回傳空 Set

def save_headlines_to_json(all_urls_set, filepath=TEMP_JSON_FILENAME):
    """
    將所有頭條 URL (Set) 儲存回 JSON 檔案。
    """
    try:
        data = {'urls': list(all_urls_set)} # 將 Set 轉回 List 以便 JSON 序列化
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        logging.info(f"✓ JSON 暫存檔 {filepath} 已更新，目前共 {len(all_urls_set)} 筆不重複 URL。")
    except IOError as e:
        logging.error(f"🔥 寫入 JSON 檔案 {filepath} 失敗: {e}")
    except Exception as e:
        logging.error(f"🔥 儲存 JSON 時發生未知錯誤: {e}")

# ===============================================================
# Cell 4: 九大媒體爬蟲函式 (v14 修復版)
# ===============================================================

# 
# (1/9) CNA 中央社 (v13 修復版)
#
def fetch_cna_headlines():
    """(1/9) 抓取 CNA 中央社 頭條"""
    logging.info("--- (1/9) 正在抓取 CNA 中央社頭條 ---")
    base_url = 'https://www.cna.com.tw'
    # *** 修正 (A): CNA 首頁使用 JS 動態載入，改抓取 'aall.aspx' (即時新聞列表) ***
    list_url = 'https://www.cna.com.tw/list/aall.aspx'
    headlines = []
    scraped_at = get_current_utc_time()

    soup = get_soup(list_url)
    if not soup:
        logging.warning("🟡 CNA 抓取失敗。")
        return headlines

    try:
        # *** 修正 (B): 使用 'aall.aspx' 的 CSS 選擇器 ***
        # L1 (即時新聞列表)
        for item in soup.select('ul.main-list li a'):
            try:
                title = item.find('h2').get_text(strip=True) if item.find('h2') else item.get_text(strip=True)
                url = normalize_url(base_url, item.get('href'))
                if title and url and 'cna.com.tw' in url:
                    # CNA 的即時列表頁沒有 L3 或 L2 的區別，因此全部標記為 L1
                    headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'CNA'})
            except Exception as e:
                logging.warning(f"🟡 CNA 剖析列表項目錯誤: {e}")
                
        # CNA 首頁的 L3/L2 很難抓取，我們在這裡專注於 L1，確保有資料流入

    except Exception as e:
        logging.error(f"🔥 CNA 爬蟲執行時發生未預期錯誤: {e}")

    logging.info(f"💡 CNA 抓取到 {len(headlines)} 筆頭條")
    return headlines

#
# (2/9) UDN 聯合新聞網 (v13 修復版)
#
def fetch_udn_headlines():
    """(2/9) 抓取 UDN 聯合新聞網 頭條"""
    logging.info("--- (2/9) 正在抓取 UDN 聯合新聞網頭條 ---")
    base_url = 'https://udn.com'
    headlines = []
    scraped_at = get_current_utc_time()

    # *** 修正 (C): UDN 的 /news/breaknews 使用動態載入，改為直接抓取首頁 ***
    soup = get_soup(base_url)
    if not soup:
        logging.warning("🟡 UDN 抓取失敗。")
        return headlines

    try:
        # *** 修正 (D): 更新 L3 選擇器 (首頁大圖) (2025/11/10) ***
        # 抓取 `div.story-list__image` 和 `div.story-list__text` 下的 a 標籤
        for item in soup.select('div.story-list__image a, div.story-list__text h2 a'):
            try:
                title = item.get('title')
                if not title:
                    # 備用方案：抓取 <a> 標籤內的文字
                    title = item.get_text(strip=True)
                
                url = normalize_url(base_url, item.get('href'))
                
                if title and url and 'udn.com' in url and '/event/' not in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'UDN'})
            except Exception as e:
                logging.warning(f"🟡 UDN Level 3 剖析錯誤: {e}")

        # *** 修正 (E): 更新 L1 選擇器 (首頁-即時新聞) (2025/11/10) ***
        # 抓取 `section.section-breaknews` (即時新聞區塊)
        for item in soup.select('div.context-box__content a'):
            try:
                title = item.get('title')
                if not title:
                    title = item.get_text(strip=True)
                
                url = normalize_url(base_url, item.get('href'))
                
                if title and url and 'udn.com' in url and '/event/' not in url:
                    # 首頁的即時新聞歸類為 L1
                    headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'UDN'})
            except Exception as e:
                logging.warning(f"🟡 UDN Level 1 剖析錯誤: {e}")
                
    except Exception as e:
        logging.error(f"🔥 UDN 爬蟲執行時發生未預期錯誤: {e}")

    logging.info(f"💡 UDN 抓取到 {len(headlines)} 筆頭條")
    return headlines

#
# (3/9) SETN 三立新聞網 (v12 成功版)
#
def fetch_setn_headlines():
    """(3/9) 抓取 SETN 三立新聞網 頭條"""
    logging.info("--- (3/9) 正在抓取 SETN 三立新聞網頭條 ---")
    base_url = 'https://www.setn.com'
    headlines = []
    scraped_at = get_current_utc_time()

    soup = get_soup(base_url, encoding='utf-8')
    if not soup:
        logging.warning("🟡 SETN 抓取失敗。")
        return headlines

    try:
        # L3 (首頁大圖)
        for item in soup.select('div.col-sm-8.top-news a[href^="/News.aspx"]'):
            try:
                title_tag = item.find('h1') or item.find('h2') or item.find('h3')
                title = title_tag.get_text(strip=True) if title_tag else None
                url = normalize_url(base_url, item.get('href'))
                if title and url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'SETN'})
            except Exception as e:
                logging.warning(f"🟡 SETN Level 3 剖析錯誤: {e}")

        # L1 (即時清單)
        for item in soup.select('div#gtMain-RealTime-News-scroll ul li a[href^="/News.aspx"]'):
            try:
                title_tag = item.find('p.realtime-list-title')
                title = title_tag.get_text(strip=True) if title_tag else item.get_text(strip=True)
                url = normalize_url(base_url, item.get('href'))
                if title and url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'SETN'})
            except Exception as e:
                logging.warning(f"🟡 SETN Level 1 剖析錯誤: {e}")
                
    except Exception as e:
        logging.error(f"🔥 SETN 爬蟲執行時發生未預期錯誤: {e}")

    logging.info(f"💡 SETN 抓取到 {len(headlines)} 筆頭條")
    return headlines

#
# (4/9) TVBS 新聞網 (v12 成功版)
#
def fetch_tvbs_headlines():
    """(4/9) 抓取 TVBS 新聞網 頭條"""
    logging.info("--- (4/9) 正在抓取 TVBS 新聞網頭條 ---")
    base_url = 'https://news.tvbs.com.tw'
    headlines = []
    scraped_at = get_current_utc_time()
    
    soup = get_soup(base_url)
    if not soup:
        logging.warning("🟡 TVBS 抓取失敗。")
        return headlines

    try:
        # L3 (首頁-大圖輪播)
        for item in soup.select('div.focus_slider ul.slick-slider li.slick-slide a'):
            try:
                title = item.find('h2').get_text(strip=True) if item.find('h2') else None
                url = normalize_url(base_url, item.get('href'))
                if title and url and 'tvbs.com.tw' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'TVBS'})
            except Exception as e:
                logging.warning(f"🟡 TVBS Level 3 剖析錯誤: {e}")

        # L2 (首頁-編輯精選)
        for item in soup.select('div.editor_slide ul.slick-slider li.slick-slide a'):
            try:
                title = item.find('h2').get_text(strip=True) if item.find('h2') else None
                url = normalize_url(base_url, item.get('href'))
                if title and url and 'tvbs.com.tw' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 2, 'scraped_at': scraped_at, 'source': 'TVBS'})
            except Exception as e:
                logging.warning(f"🟡 TVBS Level 2 剖析錯誤: {e}")

        # L1 (首頁-即時新聞)
        for item in soup.select('div.realtime_news ul.news_list li a'):
            try:
                title = item.find('h2').get_text(strip=True) if item.find('h2') else None
                url = normalize_url(base_url, item.get('href'))
                if title and url and 'tvbs.com.tw' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'TVBS'})
            except Exception as e:
                logging.warning(f"🟡 TVBS Level 1 剖析錯誤: {e}")

    except Exception as e:
        logging.error(f"🔥 TVBS 爬蟲執行時發生未預期錯誤: {e}")

    logging.info(f"💡 TVBS 抓取到 {len(headlines)} 筆頭條")
    return headlines

#
# (5/9) EBC 東森新聞 (v12 成功版)
#
def fetch_ebc_headlines():
    """(5/9) 抓取 EBC 東森新聞 頭條"""
    logging.info("--- (5/9) 正在抓取 EBC 東森新聞頭條 ---")
    base_url = 'https://news.ebc.net.tw'
    headlines = []
    scraped_at = get_current_utc_time()

    soup = get_soup(base_url)
    if not soup:
        logging.warning("🟡 EBC 抓取失敗。")
        return headlines

    try:
        # L3 (首頁-大圖輪播)
        for item in soup.select('div.news-block-style21 a.large'):
            try:
                title = item.get('title')
                url = normalize_url(base_url, item.get('href'))
                if title and url and 'ebc.net.tw' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'EBC'})
            except Exception as e:
                logging.warning(f"🟡 EBC Level 3 剖析錯誤: {e}")
                
        # L2 (首頁-大圖旁次要新聞)
        for item in soup.select('div.news-block-style21 a.small'):
            try:
                title = item.get('title')
                url = normalize_url(base_url, item.get('href'))
                if title and url and 'ebc.net.tw' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 2, 'scraped_at': scraped_at, 'source': 'EBC'})
            except Exception as e:
                logging.warning(f"🟡 EBC Level 2 剖析錯誤: {e}")

        # L1 (首頁-即時新聞)
        for item in soup.select('div.news-list-box.style1 div.news-list-block ul li a'):
            try:
                title = item.find('span.title').get_text(strip=True) if item.find('span.title') else item.get('title')
                url = normalize_url(base_url, item.get('href'))
                if title and url and 'ebc.net.tw' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'EBC'})
            except Exception as e:
                logging.warning(f"🟡 EBC Level 1 剖析錯誤: {e}")

    except Exception as e:
        logging.error(f"🔥 EBC 爬蟲執行時發生未預期錯誤: {e}")

    logging.info(f"💡 EBC 抓取到 {len(headlines)} 筆頭條")
    return headlines

#
# (6/9) NOWnews 今日新聞 (v12 成功版)
#
def fetch_nownews_headlines():
    """(6/9) 抓取 NOWnews 今日新聞 頭條"""
    logging.info("--- (6/9) 正在抓取 NOWnews 今日新聞頭條 ---")
    base_url = 'https://www.nownews.com'
    headlines = []
    scraped_at = get_current_utc_time()

    soup = get_soup(base_url)
    if not soup:
        logging.warning("🟡 NOWnews 抓取失敗。")
        return headlines

    try:
        # L3 (首頁-大圖輪播)
        for item in soup.select('div.big-slide-news a.item-text'):
            try:
                title = item.get_text(strip=True)
                url = normalize_url(base_url, item.get('href'))
                if title and url and 'nownews.com' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'NOWnews'})
            except Exception as e:
                logging.warning(f"🟡 NOWnews Level 3 剖析錯誤: {e}")

        # L1 (即時新聞列表)
        # NOWnews 即時新聞頁面是動態載入，因此我們改抓首頁的 "最新" 區塊
        for item in soup.select('div.tab-content[data-content="latest"] div.item a'):
            try:
                title = item.find('h3').get_text(strip=True) if item.find('h3') else None
                url = normalize_url(base_url, item.get('href'))
                if title and url and 'nownews.com' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'NOWnews'})
            except Exception as e:
                logging.warning(f"🟡 NOWnews Level 1 剖析錯誤: {e}")

    except Exception as e:
        logging.error(f"🔥 NOWnews 爬蟲執行時發生未預期錯誤: {e}")
        
    logging.info(f"💡 NOWnews 抓取到 {len(headlines)} 筆頭條")
    return headlines

#
# (7/9) ETtoday 新聞雲 (v13 修復版)
#
def fetch_ettoday_headlines():
    """(7/9) 抓取 ETtoday 新聞雲 頭條"""
    logging.info("--- (7/9) 正在抓取 ETtoday 新聞雲頭條 ---")
    base_url = 'https://www.ettoday.net'
    headlines = []
    scraped_at = get_current_utc_time()

    # *** 修正 (F): ETtoday 的 /news/news-list 使用動態載入，改為直接抓取首頁 ***
    soup = get_soup(base_url)
    if not soup:
        logging.warning("🟡 ETtoday 抓取失敗。")
        return headlines

    try:
        # *** 修正 (G): 更新 L3 選擇器 (首頁-頭條) (2025/11/10) ***
        # 抓取 `div.c1` (主要頭條區) 和 `div.focus-topic` (焦點圖)
        for item in soup.select('div.c1 h2 a, div.focus-topic a'):
            try:
                title = item.get_text(strip=True)
                url = normalize_url(base_url, item.get('href'))
                
                if title and url and 'ettoday.net' in url and '/events/' not in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'ETtoday'})
            except Exception as e:
                logging.warning(f"🟡 ETtoday Level 3 剖析錯誤: {e}")

        # *** 修正 (H): 更新 L1 選擇器 (首頁-即時新聞) (2025/11/10) ***
        # 抓取 `div.part_list_2` (即時新聞列表)
        for item in soup.select('div.part_list_2 h3 a'):
            try:
                title = item.get_text(strip=True)
                url = normalize_url(base_url, item.get('href'))
                
                if title and url and 'ettoday.net' in url and '/events/' not in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'ETtoday'})
            except Exception as e:
                logging.warning(f"🟡 ETtoday Level 1 剖析錯誤: {e}")

    except Exception as e:
        logging.error(f"🔥 ETtoday 爬蟲執行時發生未預期錯誤: {e}")

    logging.info(f"💡 ETtoday 抓取到 {len(headlines)} 筆頭條")
    return headlines

#
# (8/9) ChinaTimes 中時新聞網 (v13 修復版)
#
def fetch_chinatimes_headlines():
    """(8/9) 抓取 ChinaTimes 中時新聞網 頭條"""
    logging.info("--- (8/9) 正在抓取 China Times 中時新聞網頭條 ---")
    base_url = 'https://www.chinatimes.com'
    
    # *** 修正 (S): 根據 11/10 16:20 的 HTML 原始碼，只抓取首頁 ***
    soup = get_soup(base_url) 
    if not soup:
        logging.warning("🟡 China Times 首頁抓取失敗 (可能發生 403 錯誤)。")
        return []
    
    headlines = []
    scraped_at = get_current_utc_time()

    try:
        # *** 修正 (T): 更新 L3 選擇器 (首頁輪播) (2025/11/10) ***
        for item in soup.select('div.focus-gallery div.adaptive-gallery li.item-entry h3.caption-title a'):
            try:
                title = item.get_text(separator=' ', strip=True)
                url = normalize_url(base_url, item.get('href'))
                if title and url and ('chinatimes.com' in url or 'ctwant.com' in url):
                    headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'CHINATIMES'})
            except Exception as e:
                logging.warning(f"🟡 China Times Level 3 剖析錯誤: {e}")

        # *** 修正 (U): 更新 L2 選擇器 (焦點新聞) (2025/11/10) ***
        for item in soup.select('section.focus-news ul.vertical-list li h3.title a'):
            try:
                title = item.get_text(strip=True)
                url = normalize_url(base_url, item.get('href'))
                if title and url and 'chinatimes.com' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 2, 'scraped_at': scraped_at, 'source': 'CHINATIMES'})
            except Exception as e:
                logging.warning(f"🟡 China Times Level 2 剖析錯誤: {e}")

        # *** 修正 (V): 更新 L1 選擇器 (即時 Tab) (2025/11/10) ***
        for item in soup.select('div#news-pane-1-1 ul.vertical-list li h4.title a'):
            try:
                title = item.get_text(strip=True)
                url = normalize_url(base_url, item.get('href'))
                if title and url and 'chinatimes.com' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'CHINATIMES'})
            except Exception as e:
                logging.warning(f"🟡 China Times Level 1 (realtimenews) 剖析錯誤: {e}")
                
    except Exception as e:
        logging.error(f"🔥 ChinaTimes 爬蟲執行時發生未預期錯誤: {e}")

    logging.info(f"💡 China Times 抓取到 {len(headlines)} 筆頭條")
    return headlines

#
# (9/9) LTN 自由時報 (v13 修復版)
#
def fetch_ltn_headlines():
    """(9/9) 抓取自由時報 (LTN) 頭條"""
    logging.info("--- (9/9) 正在抓取 LTN 自由時報頭條 ---")
    base_url = 'https://news.ltn.com.tw'
    headlines = []
    scraped_at = get_current_utc_time()

    try:
        # *** 修正 (W): 根據 11/10 16:20 的 HTML 原始碼，L1, L2, L3 皆可從首頁抓取 ***
        # 移除對 /list/breakingnews 的額外請求，只抓取首頁
        # LTN 首頁的 base href 是 https://www.ltn.com.tw/
        soup = get_soup('https://www.ltn.com.tw/') 
        if not soup:
            logging.warning("🟡 LTN 首頁抓取失敗 (可能發生 403 錯誤)。")
            return []

        # *** 修正 (X): 更新 L3 選擇器 (首頁大圖) (2025/11/10) ***
        for item in soup.select('div#ltn_focus a.swiper-slide'):
            try:
                title = item.get('title')
                url = normalize_url(base_url, item.get('href'))
                
                # 排除內廣 (L1ad)
                if title and url and 'ltn.com.tw' in url and not 'pv6.ltn.com.tw' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'LTN'})
            except Exception as e:
                logging.warning(f"🟡 LTN Level 3 剖析錯誤: {e}")

        # *** 修正 (Y): 更新 L2 選擇器 (編輯精選) (2025/11/10) ***
        for item in soup.select('div.news10 ul.content_list li a'):
            try:
                title = item.get('title')
                url = normalize_url(base_url, item.get('href'))
                
                # 排除內廣 (L1ad)
                if title and url and 'ltn.com.tw' in url and not 'pv6.ltn.com.tw' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 2, 'scraped_at': scraped_at, 'source': 'LTN'})
            except Exception as e:
                logging.warning(f"🟡 LTN Level 2 剖析錯誤: {e}")

        # *** 修正 (Z): 更新 L1 選擇器 (即時清單) (2025/11/10) ***
        # (原 L1 抓取 /list/breakingnews 頁面，現改為抓取首頁的即時清單)
        for item in soup.select('div.breakingnews[data-desc="即時清單"] ul.content_list li a'):
            try:
                title = item.get('title')
                if not title:
                    # 備用方案：尋找 h3 標籤
                    h3_tag = item.find('h3')
                    if h3_tag:
                        title = h3_tag.get_text(strip=True)
                
                url = normalize_url(base_url, item.get('href'))
                
                if title and url and 'ltn.com.tw' in url and not 'pv6.ltn.com.tw' in url:
                    headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'LTN'})
            except Exception as e:
                logging.warning(f"🟡 LTN Level 1 剖析錯誤: {e}")

    except Exception as e:
        logging.error(f"🔥 LTN 爬蟲執行時發生未預期錯誤: {e}")

    logging.info(f"💡 LTN 抓取到 {len(headlines)} 筆頭條")
    return headlines


# ===============================================================
# Cell 5: Part 1 執行函式 (v14 全新資料集版)
# ===============================================================

def run_part1_headline_fetch():
    """
    執行 Part 1：抓取所有媒體的頭條，並找出新 URL。
    v14 版：會強制清空舊快取，抓取所有當前新聞。
    """
    logging.info("--- Part 1 開始：抓取所有媒體頭條 ---")
    
    # 讀取舊的 URL Set
    # *** v14 變更：強制使用空的 Set，忽略舊的 JSON 快取 ***
    old_urls_set = set()
    logging.info(f"v14 全新資料集模式：已清空舊快取，將重新抓取所有可見頭條。")
    
    # 存放所有新抓到的頭條 (List of Dicts)
    all_new_headlines = []
    # 存放所有這次抓到的 URL (用來更新 JSON)
    current_urls_set = set()

    # 依序執行各家爬蟲
    # 順序依照您先前的重要性排序
    fetch_functions = [
        fetch_cna_headlines,
        fetch_udn_headlines,
        fetch_setn_headlines,
        fetch_tvbs_headlines,
        fetch_ebc_headlines,
        fetch_nownews_headlines,
        fetch_ettoday_headlines,
        fetch_ltn_headlines,
        fetch_chinatimes_headlines
    ]

    for func in fetch_functions:
        try:
            headlines = func() # 執行爬蟲，回傳 List[Dict]
            
            # 找出「新的」頭條
            new_count = 0
            for h in headlines:
                url = h['url']
                current_urls_set.add(url) # 加入「本次」抓取的 Set
                
                if url not in old_urls_set:
                    all_new_headlines.append(h) # 這是一筆全新的 URL
                    new_count += 1
                    
            logging.info(f"✓ {func.__name__} 完成，發現 {new_count} 筆新頭條。")
            
        except Exception as e:
            logging.error(f"🔥 執行 {func.__name__} 時發生嚴重錯誤: {e}")
            
        time.sleep(1) # 禮貌性延遲 1 秒

    # 合併「舊的」URL Set 和「本次」抓取的 URL Set
    # 在 v14 中，old_urls_set 永遠是空的，所以 combined_urls_set == current_urls_set
    combined_urls_set = old_urls_set.union(current_urls_set)
    
    # 儲存合併後的完整 URL Set 回 JSON
    # *** v14 變更：會寫入到 news_dataset_cache.json ***
    save_headlines_to_json(combined_urls_set, TEMP_JSON_FILENAME)

    logging.info("--- Part 1 完成 ---")
    logging.info(f"總共發現 {len(all_new_headlines)} 筆全新 URL (來自9家媒體)。")
    
    return all_new_headlines # 回傳「全新」的頭條列表，給 Part 2 使用

# ===============================================================
# Cell 6: Part 2 核心功能 (抓取內文並建立快照) (v13 版)
# ===============================================================

def get_article_content(url, source):
    """
    根據不同媒體來源，抓取文章內文。
    """
    soup = get_soup(url)
    if not soup:
        return f"錯誤：抓取失敗 {url}"

    content = ""
    selectors = []

    try:
        if source == 'CNA':
            selectors = ['div.article-content p', 'div.central-text p']
        elif source == 'UDN':
            selectors = ['section.article-content__editor p']
        elif source == 'SETN':
            selectors = ['div#Content1 p']
        elif source == 'TVBS':
            selectors = ['div.article_content p', 'div.article_content div']
        elif source == 'EBC':
            selectors = ['div.raw-style p']
        elif source == 'NOWnews':
            selectors = ['article.article-content p']
        elif source == 'ETtoday':
            selectors = ['div.story p']
        elif source == 'LTN':
            selectors = ['div.text p']
        elif source == 'CHINATIMES':
            selectors = ['div.article-body p']
        
        elements = soup.select(', '.join(selectors))
        
        if not elements:
            # 如果主要選擇器抓不到，嘗試通用的 <p> 標籤
            elements = soup.find_all('p')
            if not elements:
                 return f"錯誤：找不到任何 <p> 標籤 {url}"
                 
        for el in elements:
            # 排除廣告或不相關的 <p>
            text = el.get_text(strip=True)
            if text and '延伸閱讀' not in text and '記者' not in text[:5] and '▲' not in text[:5]:
                content += text + "\n"
        
        return content if content else f"錯誤：內容為空 {url}"

    except Exception as e:
        logging.warning(f"🟡 剖析內文失敗: {url} ({e})")
        return f"錯誤：剖析失敗 {url} ({e})"

def create_snapshot_data(headline_list):
    """
    接收 Part 1 產出的「新頭條列表」，逐一抓取內文，
    並轉換為準備寫入 CSV 的 DataFrame。
    """
    snapshot_data = [] # 存放所有要寫入 CSV 的資料 (List of Dicts)
    
    if not headline_list:
        return pd.DataFrame(columns=SNAPSHOT_COLUMNS) # 回傳空的 DF

    total = len(headline_list)
    logging.info(f"--- Part 2 開始：準備抓取 {total} 筆新文章內文 ---")

    for i, headline in enumerate(headline_list):
        url = headline['url']
        source = headline['source']
        
        logging.info(f"  ({i+1}/{total}) 正在抓取 [{source}] {url}...")
        
        content = get_article_content(url, source)
        
        # 產生 URL Hash (您 CSV 中的 url_hash)
        # 這裡我們用簡單的 hash() 轉字串
        url_hash = str(hash(url))

        # 建立一筆快照資料 (7個欄位)
        snapshot_entry = {
            'url': url,
            'title': headline['title'],
            'headline_level': headline['headline_level'],
            'snapshot_at': headline['scraped_at'],
            'full_content': content,
            'source': source,
            'url_hash': url_hash
        }
        snapshot_data.append(snapshot_entry)
        
        time.sleep(0.5) # 禮貌性延遲 0.5 秒

    logging.info(f"✓ Part 2 內文抓取完成。")
    
    # 轉換為 DataFrame
    df = pd.DataFrame(snapshot_data, columns=SNAPSHOT_COLUMNS)
    return df

# ===============================================================
# Cell 7: Part 2 執行函式 (v14 全新資料集版)
# ===============================================================

def run_part2_scrape_and_append():
    """
    執行 Part 2：
    1. 執行 Part 1 取得「新」聞列表 (v14 版會抓取所有新聞)
    2. 抓取「新」聞的內文
    3. 將這些「新」聞寫入到 `news_dataset.csv`
    """
    
    # 1. 執行 Part 1
    # new_headlines_list 是一個 List[Dict]，包含所有被抓到的 URL
    new_headlines_list = run_part1_headline_fetch()
    
    if not new_headlines_list:
        logging.info("👍 Part 2 完成：沒有發現需要抓取內文的新文章。")
        return

    # 2. 抓取新文章內文
    new_df = create_snapshot_data(new_headlines_list)
    new_snapshot_data = new_df.to_dict('records') # 轉換回 List[Dict]

    # 3. 寫入 (Write) 到 CSV
    # *** v14 變更：DB_FILENAME 已被改為 'news_dataset.csv' ***
    if new_snapshot_data:
        try:
            # 檢查 DB_FILENAME 是否已存在
            file_exists = os.path.exists(DB_FILENAME)
            
            # *** v14 變更：第一次執行 'news_dataset.csv' 時，file_exists 會是 False ***
            if not file_exists:
                logging.warning(f"🟡 {DB_FILENAME} 不存在。將自動建立新檔案並寫入 7 欄位標頭。")
            else:
                logging.warning(f"🟡 {DB_FILENAME} 已存在。本次抓取的資料將「附加」到現有檔案中。")
                logging.warning(f"   如需「完全重新」的資料集，請先手動刪除 {DB_FILENAME} 和 {TEMP_JSON_FILENAME}。")

            
            # *** 使用 'a' (append) 模式 ***
            # 這樣設計可確保：
            # 1. 如果 news_dataset.csv 不存在 -> 建立新檔 + 寫入標頭 + 寫入資料 (您要的)
            # 2. 如果 news_dataset.csv 已存在 -> 附加資料 (Crontab 下次執行時)
            new_df.to_csv(
                DB_FILENAME, 
                mode='a',                 # 'a' 代表 append (附加)
                header=not file_exists,   # 只有在檔案不存在時才寫入 header
                index=False, 
                encoding='utf-8-sig'
            )
            
            logging.info(f"✅ Part 2 完成：成功抓取並寫入 {len(new_snapshot_data)} 筆新快照到 {DB_FILENAME}")
            
        except IOError as e:
            logging.error(f"🔥 Part 2 錯誤：儲存到 {DB_FILENAME} 失敗: {e}")
        except Exception as e:
            logging.error(f"🔥 Part 2 儲存時發生未知錯誤: {e}")
    else:
        logging.warning("👍 Part 2 完成：本次快照未成功抓取到任何新文章。")

# ===============================================================
# Cell 8: 完整執行所有爬蟲任務
# ===============================================================

def main_execution():
    """
    依序執行所有爬蟲步驟
    """
    
    start_time = time.time()
    logging.info(f"======= 爬蟲任務開始於 {get_current_utc_time()} =======")
    
    # 步驟一 & 步驟二：
    # run_part2_scrape_and_append() 會自動呼叫 Part 1，
    # 然後抓取新文章內文，最後寫入到 news_dataset.csv
    run_part2_scrape_and_append()
    
    # 步驟三：(已移除) 
    
    end_time = time.time()
    sources = "CNA, UDN, SETN, TVBS, EBC, NOWnews, ETtoday, LTN, ChinaTimes"
    logging.info(f"\n======= 爬蟲任務完成 ({sources})，總耗時: {end_time - start_time:.2f} 秒 =======")

# 確保這個 .py 檔案被直接執行時，才會觸發 main_execution()
if __name__ == "__main__":
    # 立即執行主程式
    main_execution()